# 3차 제출 재현
quantile-ensemble LightGBM (alpha 0.70/0.75/0.80 × seed), post-scale 없음 + IDW 공간 feature.

> 참고: 제출 이후 `features2`의 temporal feature가 제거되어, 현재 코드는 `data/3차.csv`를 **근사** 재현합니다 (상관 ≈ 0.998, 약 2% 차). 2차와 동일한 모델이며 post-scale(×1.05/1.08)만 적용하지 않은 버전입니다.

## 1. 라이브러리 · 상수 정의

**무엇을** — 넘파이/판다스와 **LightGBM**을 불러오고 공통 상수를 정의합니다. 3차는 2차와 **동일한 모델·피처**를 쓰되, 그룹별 후보정(post-scale ×1.05/1.08)만 적용하지 않은 버전입니다.

**왜 / 어떻게**
- `TARGET_COLS` : 예측 대상 3개 발전 그룹.
- `CAPACITY_KWH` : 그룹별 설비용량. 모델은 **용량계수(CF, 0~1)** 를 학습하고 예측 후 용량을 곱해 kWh로 복원.
- `HUB = 117.0` : 허브 높이(m), 풍속 외삽 기준.

In [1]:
import numpy as np, pandas as pd
import lightgbm as lgb

D = "../data/"
TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
HUB = 117.0

## 2. 피처 엔지니어링 — 공통 피처 + IDW 공간 피처 (2차와 동일)

원시 기상격자를 예보 시각당 1행 피처로 변환합니다. 2차와 완전히 같은 피처 구성입니다.

**행 단위 파생**
- `_ldaps_rowfeats` / `_gfs_rowfeats` : `hypot(u,v)` 풍속 합성, max−min 돌풍, 멱법칙 허브풍속 외삽(`ws_hub=ws50·(117/50)^alpha`), 세제곱항(발전량 ∝ v³), 공기밀도(`p/RT`) 등.
- `calendar_features` : 시각·월·연중일 sin/cos 인코딩.

**IDW 공간 피처** — 발전 그룹 위치가 제각각이므로 격자점들을 **그룹 중심점에 가까울수록 크게 가중**해 합성.
- `_idw_weights` : 위경도를 km로 환산 후 거리제곱의 역수 `1/(d²+0.25)`를 가중치로 사용.
- `_time_pivot`/`_idw_series` : (시각×격자) 표를 만들어 가중치와 내적 → 시각별 IDW 합성값.
- `build_shared` : 전 그룹 공유 집계(평균/최대/표준편차) 피처.
- `build_group_extra` : 그룹별 IDW 풍속·풍향(sin/cos)·상호작용·GFS 100 m 풍속 등 그룹 전용 피처.

In [2]:
def _ldaps_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    u10, v10 = df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"]
    o["ws10"] = np.hypot(u10, v10)
    u50 = (df["heightAboveGround_50_50MUmax"] + df["heightAboveGround_50_50MUmin"]) / 2
    v50 = (df["heightAboveGround_50_50MVmax"] + df["heightAboveGround_50_50MVmin"]) / 2
    o["ws50"] = np.hypot(u50, v50)
    o["ws50max"] = np.hypot(df["heightAboveGround_50_50MUmax"], df["heightAboveGround_50_50MVmax"])
    o["ws50_gust"] = (df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]).abs() \
        + (df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]).abs()
    o["blws"] = np.hypot(df["heightAboveGround_5_XBLWS"], df["heightAboveGround_5_YBLWS"])
    alpha = np.log(np.clip(o["ws50"], 0.1, None) / np.clip(o["ws10"], 0.1, None)) / np.log(50 / 10)
    alpha = alpha.clip(-0.5, 1.0)
    o["ws_hub"] = o["ws50"] * (HUB / 50.0) ** alpha
    o["ws_hub3"] = o["ws_hub"] ** 3
    o["ws50_3"] = o["ws50"] ** 3
    o["rho"] = df["surface_0_sp"] / (287.05 * (df["heightAboveGround_2_t"]))
    o["t2"] = df["heightAboveGround_2_t"]
    o["sp"] = df["surface_0_sp"]
    o["blh"] = df["etc_0_blh"]
    o["u50"], o["v50"] = u50, v50
    return o


def _gfs_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    o["gws10"] = np.hypot(df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"])
    o["gws80"] = np.hypot(df["heightAboveGround_80_u"], df["heightAboveGround_80_v"])
    o["gws100"] = np.hypot(df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"])
    o["gws100_3"] = o["gws100"] ** 3
    o["gust"] = df["surface_0_gust"]
    o["gpbl"] = np.hypot(df["planetaryBoundaryLayer_0_u"], df["planetaryBoundaryLayer_0_v"])
    o["gws850"] = np.hypot(df["isobaricInhPa_850_u"], df["isobaricInhPa_850_v"])
    o["gu100"], o["gv100"] = df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"]
    return o


def calendar_features(idx):
    dt = pd.to_datetime(pd.Series(idx))
    out = pd.DataFrame(index=idx)
    h, m, doy = dt.dt.hour.values, dt.dt.month.values, dt.dt.dayofyear.values
    out["hour_sin"] = np.sin(2 * np.pi * h / 24);  out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["month_sin"] = np.sin(2 * np.pi * m / 12); out["month_cos"] = np.cos(2 * np.pi * m / 12)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365); out["doy_cos"] = np.cos(2 * np.pi * doy / 365)
    return out


GROUP_CENTROID = {
    "kpx_group_1": (37.28713, 128.95202),
    "kpx_group_2": (37.28225, 128.96515),
    "kpx_group_3": (37.27520, 128.97144),
}


def _grid_coords(df):
    return df.drop_duplicates("grid_id").set_index("grid_id")[["latitude", "longitude"]].sort_index()


def _idw_weights(gridc, clat, clon, power=2.0):
    dy = (gridc["latitude"] - clat) * 111.0
    dx = (gridc["longitude"] - clon) * 111.0 * np.cos(np.radians(clat))
    d2 = dx**2 + dy**2
    w = 1.0 / (d2 + 0.25)
    w = w ** (power / 2.0)
    return (w / w.sum())


def _time_pivot(rowfeats, times, gridids, col):
    rf = pd.DataFrame({"t": times.values, "g": gridids.values, "v": rowfeats[col].values})
    return rf.pivot_table(index="t", columns="g", values="v")


def _idw_series(pivot, weights):
    w = weights.reindex(pivot.columns).fillna(0.0).values
    return pd.Series(pivot.values @ w, index=pivot.index)


def build_shared(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)

    def agg(rf, times, prefix):
        x = rf.copy(); x["t"] = times.values; g = x.groupby("t")
        m = g.mean(); m.columns = [f"{prefix}_{c}_mean" for c in m.columns]
        mx = g.max(); mx = mx[[c for c in mx.columns if c != "t"]]
        mx.columns = [f"{prefix}_{c}_max" for c in mx.columns]
        sd = g.std(); sd.columns = [f"{prefix}_{c}_std" for c in sd.columns]
        keep_mx = [c for c in mx.columns if "ws" in c or "gust" in c]
        keep_sd = [c for c in sd.columns if "ws" in c]
        return pd.concat([m, mx[keep_mx], sd[keep_sd]], axis=1)

    L = agg(lrf, lt, "l"); G = agg(grf, gt, "g")
    feat = L.join(G, how="left")
    feat = feat.join(calendar_features(feat.index), how="left")
    return feat


def build_group_extra(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)
    lgc = _grid_coords(ldaps); ggc = _grid_coords(gfs)
    piv = {c: _time_pivot(lrf, lt, ldaps["grid_id"], c) for c in ["ws_hub", "ws_hub3", "ws50", "u50", "v50", "ws50_gust"]}
    gpiv = {c: _time_pivot(grf, gt, gfs["grid_id"], c) for c in ["gws100", "gu100", "gv100"]}
    out = {}
    for tgt, (clat, clon) in GROUP_CENTROID.items():
        lw = _idw_weights(lgc, clat, clon)
        gw = _idw_weights(ggc, clat, clon)
        e = pd.DataFrame(index=piv["ws_hub"].index)
        e["gws_hub"] = _idw_series(piv["ws_hub"], lw)
        e["gws_hub3"] = _idw_series(piv["ws_hub3"], lw)
        e["gws50"] = _idw_series(piv["ws50"], lw)
        e["ggust"] = _idw_series(piv["ws50_gust"], lw)
        u = _idw_series(piv["u50"], lw); v = _idw_series(piv["v50"], lw)
        ang = np.arctan2(v, u)
        e["gwd_sin"] = np.sin(ang); e["gwd_cos"] = np.cos(ang)
        e["gws_x_sin"] = e["gws50"] * e["gwd_sin"]; e["gws_x_cos"] = e["gws50"] * e["gwd_cos"]
        e["ggfs100"] = _idw_series(gpiv["gws100"], gw)
        out[tgt] = e
    return out

## 3. 데이터 로드 · 피처 생성

**무엇을 / 왜** — 학습·테스트 원시 데이터·라벨·제출양식을 읽고, 공통 피처(`shared`)와 그룹별 IDW 피처(`extra`)를 각각 생성합니다. 모든 피처의 인덱스를 예보 시각(datetime)으로 통일해 라벨/제출 시각과 정확히 정렬합니다. (2차와 동일.)

In [3]:
ldaps = pd.read_csv(D+"train/ldaps_train.csv"); gfs = pd.read_csv(D+"train/gfs_train.csv")
lab = pd.read_csv(D+"train/train_labels.csv"); lab["kst_dtm"] = pd.to_datetime(lab["kst_dtm"])
shared = build_shared(ldaps, gfs); shared.index = pd.to_datetime(shared.index)
extra = build_group_extra(ldaps, gfs)
for k in extra: extra[k].index = pd.to_datetime(extra[k].index)
lab = lab.set_index("kst_dtm")

ldaps_te = pd.read_csv(D+"test/ldaps_test.csv"); gfs_te = pd.read_csv(D+"test/gfs_test.csv")
sub = pd.read_csv(D+"sample_submission.csv"); sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"])
shared_te = build_shared(ldaps_te, gfs_te); shared_te.index = pd.to_datetime(shared_te.index)
extra_te = build_group_extra(ldaps_te, gfs_te)
for k in extra_te: extra_te[k].index = pd.to_datetime(extra_te[k].index)

## 4. 분위수 앙상블 학습 · 예측 · 제출 (post-scale 없음)

**무엇을** — 그룹별로 LightGBM 분위수 회귀를 앙상블 학습해 제출 파일을 만듭니다. **2차와 유일한 차이는 후보정(post-scale)을 적용하지 않는다**는 점입니다.

**왜 / 어떻게**
- `kw(a, s)` : `objective="quantile", alpha=a` 분위수 회귀. `alpha` 0.70~0.80으로 중앙값보다 약간 높게 예측.
- `pred_g` : `ALPHAS`(0.70/0.75/0.80) × `SEEDS`(42/7/2024)의 모든 조합을 학습·평균 → 분위수-시드 앙상블로 편향·분산 완화.
- 2차의 `SCALE`(×1.05/1.08) 곱셈이 **없음** → CF에 용량만 곱해 kWh로 복원 후 `[0, 용량]` 클리핑. 즉 후보정의 효과를 배제한 "순수 앙상블" 버전입니다.
- 결과는 `utf-8-sig`로 저장.

In [4]:
ALPHAS = [0.70, 0.75, 0.80]; SEEDS = [42, 7, 2024]

def kw(a, s):
    return dict(objective="quantile", alpha=a, n_estimators=1200, learning_rate=0.03,
        num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=1.0, random_state=s, n_jobs=-1, verbose=-1)

def Xg(t, idx, S, E):
    return S.reindex(idx).join(E[t].reindex(idx), how="left")

def pred_g(Xtr, ytr, Xva):
    return np.mean([np.clip(lgb.LGBMRegressor(**kw(a, s)).fit(Xtr, ytr).predict(Xva), 0, 1)
                    for a in ALPHAS for s in SEEDS], axis=0)

tidx = sub["forecast_kst_dtm"].values
out = sub[["forecast_id", "forecast_kst_dtm"]].copy()
for t in TARGET_COLS:
    m = lab[t].notna(); idx = lab.index[m]
    out[t] = pred_g(Xg(t, idx, shared, extra), (lab.loc[idx, t] / CAPACITY_KWH[t]).values, Xg(t, tidx, shared_te, extra_te)) * CAPACITY_KWH[t]
    out[t] = np.clip(out[t], 0, CAPACITY_KWH[t])

out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
out.to_csv(D+"submission_3차.csv", index=False, encoding="utf-8-sig")
out.head()

,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,20160.795573,19628.213638,17797.143777
1,forecast_0002,2025-01-01 02:00:00,19566.469335,19585.696287,17506.856594
2,forecast_0003,2025-01-01 03:00:00,18888.269874,19545.763754,17926.913400
3,forecast_0004,2025-01-01 04:00:00,18446.032870,19302.466003,17112.540603
4,forecast_0005,2025-01-01 05:00:00,18805.573101,19450.568805,17543.460399


## 5. 근사 재현 검증

**무엇을 / 왜** — 생성한 예측을 기존 제출 `data/3차.csv`와 상관계수·최대오차로 비교합니다. 제출 이후 temporal feature가 제거돼 완전 일치는 아니며(상관 ≈ 0.998, 약 2% 차), 근사 재현임을 정량적으로 확인합니다.

In [5]:
ref = pd.read_csv(D+"3차.csv")
for t in TARGET_COLS:
    corr = np.corrcoef(out[t], ref[t])[0, 1]
    md = np.max(np.abs(out[t].values - ref[t].values))
    print(f"{t}: corr={corr:.4f}  maxdiff={md:.1f}")

kpx_group_1: corr=0.9976  maxdiff=3074.3
kpx_group_2: corr=0.9975  maxdiff=3641.0
kpx_group_3: corr=0.9977  maxdiff=2682.2
